In [1]:
print("Nama  : Farid Hilmy Cordova")
print("NPM   : 10124442")
print("Kelas : 3KA31")
print(" ")
print("Keterangan : Ini adalah hasil manipulasi dataset tanpa melewati proses pembersihan data")

Nama  : Farid Hilmy Cordova
NPM   : 10124442
Kelas : 3KA31
 
Keterangan : Ini adalah hasil manipulasi dataset tanpa melewati proses pembersihan data


In [2]:
import pandas as pd
import numpy as np

# Menampilkan angka desimal dengan pemisah ribuan agar tidak muncul notasi ilmiah (mis. 1.256382e+08)
pd.options.display.float_format = '{:,.2f}'.format

df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_transaksi   5030 non-null   object
 1   tanggal        5030 non-null   object
 2   kode_gerai     5030 non-null   object
 3   kode_kasir     4769 non-null   object
 4   kode_produk    5030 non-null   object
 5   jumlah         5030 non-null   int64 
 6   harga_satuan   5030 non-null   object
 7   diskon_persen  5030 non-null   int64 
 8   status         5030 non-null   object
dtypes: int64(2), object(7)
memory usage: 353.8+ KB


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


# Task 1 — Indexing & Selection (Pembersihan Anomali & Imputasi)

**1. Pembersihan Format Harga**

In [3]:
# Tipe data harga_satuan sebelum dibersihkan (terbaca sebagai objek/teks)
print(df['harga_satuan'].dtype)
print(df.loc[~df['harga_satuan'].str.fullmatch(r'\d+'), 'harga_satuan'].unique()[:10])

object
['7500,00' '11000,00' '13.000' '27000,00' '13500,00' '28.000' '43500,00'
 '16500,00' '4500,00' '11.000']


In [4]:
# Nilai seperti '13.000' memakai titik sebagai pemisah ribuan, bukan desimal,
# sehingga titiknya dihapus terlebih dahulu agar tidak terbaca 13.0
ribuan = df['harga_satuan'].str.fullmatch(r'\d{1,3}\.\d{3}')
df.loc[ribuan, 'harga_satuan'] = df.loc[ribuan, 'harga_satuan'].str.replace('.', '', regex=False)

# Mengubah koma desimal menjadi titik (contoh: '7500,00' -> '7500.00'), lalu konversi ke float
df['harga_satuan'] = df['harga_satuan'].str.replace(',', '.', regex=False).astype(float)

print(df['harga_satuan'].dtype)
df['harga_satuan'].describe()

float64


,harga_satuan
count,"5,030.00"
mean,"22,185.09"
std,"15,205.70"
min,"4,000.00"
25%,"11,000.00"
50%,"16,000.00"
75%,"32,000.00"
max,"54,000.00"


**2. Penanganan Kuantitas Negatif**

In [5]:
print('Jumlah negatif sebelum:', (df['jumlah'] < 0).sum())

# Boolean indexing dengan .loc[] untuk mengubah jumlah negatif menjadi positif
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

print('Jumlah negatif sesudah:', (df['jumlah'] < 0).sum())

Jumlah negatif sebelum: 41
Jumlah negatif sesudah: 0


**3. Imputasi Nilai Kosong Kasir**

In [6]:
print('NaN kode_kasir sebelum:', df['kode_kasir'].isna().sum())

# Mengisi nilai kosong kode_kasir dengan label default
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print('NaN kode_kasir sesudah:', df['kode_kasir'].isna().sum())

NaN kode_kasir sebelum: 261
NaN kode_kasir sesudah: 0


**4. Seleksi Posisi Slicing**

In [7]:
# 5 baris pertama dengan kolom id_transaksi, tanggal, kode_gerai, dan jumlah
df.iloc[:5, [0, 1, 2, 5]]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


# Task 2 — Operasi Aritmatika (Feature Engineering)

**1. Perhitungan Omzet Kotor**

In [8]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df[['id_transaksi', 'harga_satuan', 'jumlah', 'Total_Kotor']].head()

,id_transaksi,harga_satuan,jumlah,Total_Kotor
0,TRX0241065,"13,000.00",4,"52,000.00"
1,TRX0240510,"21,000.00",1,"21,000.00"
2,TRX0244483,"7,500.00",3,"22,500.00"
3,TRX0242532,"6,500.00",2,"13,000.00"
4,TRX0243509,"6,000.00",1,"6,000.00"


**2. Perhitungan Diskon**

In [9]:
# diskon_persen berskala 0-100, sehingga dibagi 100 terlebih dahulu
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df[['id_transaksi', 'Total_Kotor', 'diskon_persen', 'Nominal_Diskon']].head()

,id_transaksi,Total_Kotor,diskon_persen,Nominal_Diskon
0,TRX0241065,"52,000.00",0,0.00
1,TRX0240510,"21,000.00",0,0.00
2,TRX0244483,"22,500.00",0,0.00
3,TRX0242532,"13,000.00",0,0.00
4,TRX0243509,"6,000.00",0,0.00


**3. Perhitungan Omzet Bersih & Pajak**

In [10]:
# (Total_Kotor - Nominal_Diskon) * 1.11  -> sudah termasuk PPN 11%
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11
df[['id_transaksi', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,id_transaksi,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,"52,000.00",0.00,"57,720.00"
1,TRX0240510,"21,000.00",0.00,"23,310.00"
2,TRX0244483,"22,500.00",0.00,"24,975.00"
3,TRX0242532,"13,000.00",0.00,"14,430.00"
4,TRX0243509,"6,000.00",0.00,"6,660.00"


# Task 3 — Filtering (Penyaringan Data)

**1. Filter Multi-Kondisi**

In [11]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]

print('Jumlah baris yang memenuhi kriteria:', len(filter_multi))
filter_multi

Jumlah baris yang memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"
...,...,...,...,...,...,...,...,...,...,...,...,...
4877,TRX0243898,2024-04-23,G05,K008,SGR1204,12,"51,000.00",0,SELESAI,"612,000.00",0.00,"679,320.00"
4911,TRX0241966,2024-04-22,G09,K024,SGR1206,3,"42,000.00",0,SELESAI,"126,000.00",0.00,"139,860.00"
4925,TRX0240377,2025-06-11,G09,K028,XXX9969,12,"51,500.00",15,SELESAI,"618,000.00","92,700.00","583,083.00"
4964,TRX0243219,2025-11-17,G09,K003,SGR1204,6,"43,500.00",0,SELESAI,"261,000.00",0.00,"289,710.00"


**2. Filter dengan Query**

In [12]:
filter_g10 = df.query('kode_gerai == "G10" and jumlah >= 5')
filter_g10

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,"52,000.00",10,SELESAI,"312,000.00","31,200.00","311,688.00"
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,"18,000.00",0,SELESAI,"108,000.00",0.00,"119,880.00"
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,"6,000.00",0,SELESAI,"36,000.00",0.00,"39,960.00"
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,"54,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,"29,000.00",0,SELESAI,"174,000.00",0.00,"193,140.00"


# Task 4 — Sorting (Pengurutan Data)

**1. Multi-Column Sorting**

In [13]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_sorted

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,"32,500.00",20,SELESAI,"47,320,000.00","9,464,000.00","42,020,160.00"
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,"50,500.00",0,SELESAI,"606,000.00",0.00,"672,660.00"
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,"47,000.00",0,SELESAI,"564,000.00",0.00,"626,040.00"
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,"42,000.00",0,SELESAI,"504,000.00",0.00,"559,440.00"
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,"45,500.00",10,SELESAI,"546,000.00","54,600.00","545,454.00"
...,...,...,...,...,...,...,...,...,...,...,...,...
3724,TRX0242116,2025-10-31,G12,K024,MIN1008,1,"4,500.00",0,SELESAI,"4,500.00",0.00,"4,995.00"
1486,TRX0241390,2024-12-29,G12,K008,PRW1300,1,"5,500.00",20,SELESAI,"5,500.00","1,100.00","4,884.00"
2291,TRX0244124,2024-01-26,G12,K035,MIN1008,1,"5,500.00",20,SELESAI,"5,500.00","1,100.00","4,884.00"
3850,TRX0242420,2024-01-29,G12,K019,RMT1406,1,"4,000.00",0,SELESAI,"4,000.00",0.00,"4,440.00"


**2. Top-N Records**

In [14]:
top_5_transaksi = df.nlargest(5, 'Total_Bersih')
top_5_transaksi

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
994,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,"48,000.00",10,SELESAI,"100,464,000.00","10,046,400.00","100,363,536.00"
4985,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,"51,500.00",10,SELESAI,"67,001,500.00","6,700,150.00","66,934,498.50"
467,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,"54,000.00",0,SELESAI,"48,168,000.00",0.00,"53,466,480.00"
3393,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,"23,000.00",10,SELESAI,"45,609,000.00","4,560,900.00","45,563,391.00"
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,"32,500.00",20,SELESAI,"47,320,000.00","9,464,000.00","42,020,160.00"


# Task 5 — Grouping & Agregasi (Pelaporan Kinerja Gerai)

**1. Laporan Agregasi Per Gerai**

In [15]:
# Menyaring hanya transaksi berstatus SELESAI
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai', as_index=False).agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

laporan_gerai

,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
10,G11,398,3170,"125,638,152.75","315,673.75"
8,G09,379,3027,"101,157,630.00","266,906.68"
5,G06,383,3103,"91,228,624.50","238,194.84"
6,G07,384,4284,"80,559,748.50","209,791.01"
0,G01,450,2635,"70,967,378.25","157,705.28"
2,G03,372,3274,"59,991,226.50","161,266.74"
9,G10,401,1742,"58,964,892.75","147,044.62"
7,G08,408,1700,"35,630,084.25","87,328.64"
11,G12,406,1010,"24,699,220.50","60,835.52"
3,G04,399,1042,"24,666,503.25","61,820.81"


**2. Transformasi Kelompok (Window Function)**

In [16]:
# (Total_Bersih / Total_Omzet_Gerai_Tersebut) * 100
total_omzet_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_omzet_gerai) * 100

# Kontribusi per transaksi nilainya kecil, sehingga ditampilkan dengan 6 desimal
with pd.option_context('display.float_format', '{:,.6f}'.format):
    display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())

# Pemeriksaan: total kontribusi tiap gerai harus tepat 100%
df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(2)

,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,"57,720.000000",0.056292
1,TRX0240510,G09,"23,310.000000",0.022733
2,TRX0244483,G11,"24,975.000000",0.018326
3,TRX0242532,G05,"14,430.000000",0.060075
4,TRX0243509,G05,"6,660.000000",0.027727


,Kontribusi_Gerai_Persen
kode_gerai,
G01,100.00
G02,100.00
G03,100.00
G04,100.00
G05,100.00
G06,100.00
G07,100.00
G08,100.00
G09,100.00
